# SNUH FERMAT Foundation ETL Pilot

Research Pod JupyterLab에서 실행하는 첫 전처리 감사 노트북입니다.

목표:

1. PostgreSQL 연결과 OMOP 권한 확인
2. 실제 DB snapshot과 핵심 테이블 규모 확인
3. 환자 단위 70/15/15 split 규칙 고정
4. DX/RX/PX/LAB/DTH/SEX 입력 가능성 감사
5. LAB의 numeric/categorical/unit 분포와 frequent/rare 기준 결정 자료 생성

이 노트북은 source CDM 테이블을 복제하거나 서버에 영구 테이블을 만들지 않습니다. 비밀번호도 파일에 저장하지 않습니다.

In [ ]:
# Pod 이미지에 패키지가 없을 때 한 번만 실행하세요.
%pip install -q "psycopg[binary]>=3" pandas pyarrow psutil


In [ ]:
import getpass
import json
import os
import platform
import subprocess
import time
from pathlib import Path

import pandas as pd
import psutil
import psycopg
from psycopg import sql

DB_HOST = "pg-2vge6u.vpc-cdb-kr.gov-ntruss.com"
DB_PORT = 5432
DB_NAME = "cdm"
DB_USER = "jaegyun_jung"
DB_SCHEMA = "cdm2024_official"
DB_SSLMODE = "disable"
DB_END_DATE = "2025-02-05"
SPLIT_SEED = 42

OUTPUT_DIR = Path("outputs/snuh_foundation_etl_pilot")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# measurement 전체 GROUP BY는 무거우므로 초기 점검 후 명시적으로 켭니다.
RUN_HEAVY_AUDIT = False

DB_PASSWORD = os.environ.get("SNUH_CDM_PASSWORD")
if not DB_PASSWORD:
    DB_PASSWORD = getpass.getpass("SNUH CDM password: ")

print("Python:", platform.python_version())
print("CPU logical cores:", psutil.cpu_count(logical=True))
print("RAM GB:", round(psutil.virtual_memory().total / 1024**3, 1))
print("Output directory:", OUTPUT_DIR.resolve())


In [ ]:
# GPU가 없는 CPU Pod에서도 정상적으로 넘어갑니다.
try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception:
    print("No NVIDIA GPU visible in this pod (expected for CPU ETL pod).")


In [ ]:
conn = psycopg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    sslmode=DB_SSLMODE,
    connect_timeout=15,
    application_name="fermat_foundation_etl_pilot",
    options="-c statement_timeout=30min",
)
conn.autocommit = True

def query_df(query, params=None):
    started = time.time()
    with conn.cursor() as cur:
        cur.execute(query, params or ())
        columns = [item.name for item in cur.description]
        rows = cur.fetchall()
    frame = pd.DataFrame(rows, columns=columns)
    print(f"Completed in {time.time() - started:,.1f}s; rows={len(frame):,}")
    return frame

identity = query_df("""
SELECT
    current_user,
    current_database(),
    version(),
    pg_backend_pid(),
    has_schema_privilege(%s, 'USAGE') AS schema_usage,
    has_schema_privilege(%s, 'CREATE') AS schema_create
""", (DB_SCHEMA, DB_SCHEMA))
display(identity)


## 1. 핵심 OMOP 테이블과 SELECT 권한

In [ ]:
CORE_TABLES = [
    "person", "observation_period", "visit_occurrence",
    "condition_occurrence", "drug_exposure", "procedure_occurrence",
    "measurement", "death", "concept", "concept_relationship",
    "concept_ancestor",
]

table_access = query_df("""
SELECT
    table_name,
    has_table_privilege(
        current_user,
        quote_ident(table_schema) || '.' || quote_ident(table_name),
        'SELECT'
    ) AS can_select
FROM information_schema.tables
WHERE table_schema = %s
  AND table_name = ANY(%s)
ORDER BY table_name
""", (DB_SCHEMA, CORE_TABLES))
display(table_access)

missing = sorted(set(CORE_TABLES) - set(table_access.table_name))
assert not missing, f"Missing core tables: {missing}"
assert table_access.can_select.all(), "Some core tables lack SELECT permission"


## 2. DB snapshot과 테이블 규모

`pg_class.reltuples`는 빠른 추정치입니다. measurement 전체 `COUNT(*)`는 이 단계에서 실행하지 않습니다.

In [ ]:
table_profile = query_df("""
SELECT
    c.relname AS table_name,
    c.reltuples::bigint AS estimated_rows,
    pg_total_relation_size(c.oid) AS total_bytes,
    ROUND(pg_total_relation_size(c.oid) / 1024.0 / 1024.0 / 1024.0, 2)
        AS total_gb
FROM pg_class AS c
JOIN pg_namespace AS n ON n.oid = c.relnamespace
WHERE n.nspname = %s
  AND c.relname = ANY(%s)
ORDER BY total_bytes DESC
""", (DB_SCHEMA, CORE_TABLES))
display(table_profile)
table_profile.to_csv(OUTPUT_DIR / "table_profile.csv", index=False)


In [ ]:
snapshot = query_df(sql.SQL("""
SELECT
    (SELECT COUNT(*) FROM {}.person) AS persons,
    (SELECT COUNT(*) FROM {}.condition_occurrence) AS condition_events,
    (SELECT MIN(observation_period_start_date)::text
       FROM {}.observation_period) AS raw_min_observation_start,
    (SELECT MAX(observation_period_end_date)::text
       FROM {}.observation_period) AS raw_max_observation_end,
    (SELECT COUNT(*)
       FROM {}.observation_period
      WHERE observation_period_start_date
                NOT BETWEEN DATE '1900-01-01' AND DATE '2100-12-31'
         OR observation_period_end_date
                NOT BETWEEN DATE '1900-01-01' AND DATE '2100-12-31'
         OR observation_period_end_date < observation_period_start_date)
        AS invalid_observation_rows
""").format(*[sql.Identifier(DB_SCHEMA) for _ in range(5)]))
display(snapshot)
snapshot.to_csv(OUTPUT_DIR / "snapshot.csv", index=False)


## 3. 환자 split 규칙 확인

환자 ID를 외부로 반출하지 않고 PostgreSQL 안에서 동일하게 재현 가능한 hash split을 사용합니다.

- train: bucket 0–69
- validation: bucket 70–84
- test: bucket 85–99

In [ ]:
split_profile = query_df(sql.SQL("""
WITH assigned AS (
    SELECT
        person_id,
        mod(
            hashtextextended(person_id::text, %s)
                & 9223372036854775807,
            100
        ) AS bucket
    FROM {}.person
)
SELECT
    CASE
        WHEN bucket < 70 THEN 'train'
        WHEN bucket < 85 THEN 'val'
        ELSE 'test'
    END AS split,
    COUNT(*) AS patients,
    MIN(bucket) AS min_bucket,
    MAX(bucket) AS max_bucket
FROM assigned
GROUP BY 1
ORDER BY 1
""").format(sql.Identifier(DB_SCHEMA)), (SPLIT_SEED,))
display(split_profile)
split_profile.to_csv(OUTPUT_DIR / "split_profile.csv", index=False)
assert split_profile.patients.sum() == int(snapshot.persons.iloc[0])


## 4. 도메인별 날짜·concept·source completeness

In [ ]:
domain_quality = query_df(sql.SQL("""
SELECT * FROM (
    SELECT
        'condition' AS domain_name,
        COUNT(*) AS rows,
        COUNT(*) FILTER (WHERE condition_start_date IS NULL) AS missing_date,
        COUNT(*) FILTER (WHERE condition_concept_id IS NULL OR condition_concept_id = 0)
            AS missing_standard_concept,
        COUNT(*) FILTER (WHERE condition_source_concept_id IS NULL
                              OR condition_source_concept_id = 0)
            AS missing_source_concept
    FROM {}.condition_occurrence
    UNION ALL
    SELECT
        'drug', COUNT(*),
        COUNT(*) FILTER (WHERE drug_exposure_start_date IS NULL),
        COUNT(*) FILTER (WHERE drug_concept_id IS NULL OR drug_concept_id = 0),
        COUNT(*) FILTER (WHERE drug_source_concept_id IS NULL
                              OR drug_source_concept_id = 0)
    FROM {}.drug_exposure
    UNION ALL
    SELECT
        'procedure', COUNT(*),
        COUNT(*) FILTER (WHERE procedure_date IS NULL),
        COUNT(*) FILTER (WHERE procedure_concept_id IS NULL
                              OR procedure_concept_id = 0),
        COUNT(*) FILTER (WHERE procedure_source_concept_id IS NULL
                              OR procedure_source_concept_id = 0)
    FROM {}.procedure_occurrence
    UNION ALL
    SELECT
        'measurement', COUNT(*),
        COUNT(*) FILTER (WHERE measurement_date IS NULL),
        COUNT(*) FILTER (WHERE measurement_concept_id IS NULL
                              OR measurement_concept_id = 0),
        COUNT(*) FILTER (WHERE measurement_source_concept_id IS NULL
                              OR measurement_source_concept_id = 0)
    FROM {}.measurement
) AS q
ORDER BY domain_name
""").format(*[sql.Identifier(DB_SCHEMA) for _ in range(4)]))
display(domain_quality)
domain_quality.to_csv(OUTPUT_DIR / "domain_quality.csv", index=False)


## 5. Measurement 결과 유형과 unit completeness

아래 셀은 measurement 전체를 한 번 scan합니다. CPU Pod에서 수 분 이상 걸릴 수 있습니다.

In [ ]:
measurement_quality = query_df(sql.SQL("""
SELECT
    COUNT(*) AS rows,
    COUNT(*) FILTER (WHERE value_as_number IS NOT NULL) AS numeric_rows,
    COUNT(*) FILTER (WHERE value_as_concept_id IS NOT NULL
                          AND value_as_concept_id <> 0) AS categorical_rows,
    COUNT(*) FILTER (WHERE value_as_number IS NULL
                          AND (value_as_concept_id IS NULL OR value_as_concept_id = 0))
        AS result_missing_rows,
    COUNT(*) FILTER (WHERE value_as_number IS NOT NULL
                          AND (unit_concept_id IS NULL OR unit_concept_id = 0))
        AS numeric_missing_unit_rows,
    COUNT(DISTINCT measurement_concept_id) AS measurement_concepts,
    COUNT(DISTINCT (measurement_concept_id, unit_concept_id))
        FILTER (WHERE value_as_number IS NOT NULL) AS numeric_test_unit_pairs
FROM {}.measurement
""").format(sql.Identifier(DB_SCHEMA)))
display(measurement_quality)
measurement_quality.to_csv(OUTPUT_DIR / "measurement_quality.csv", index=False)


## 6. LAB frequent/rare 기준 결정을 위한 heavy audit

이 셀은 `(measurement_concept_id, unit_concept_id)` 전체를 train split 기준으로 집계합니다. 앞 셀 결과와 Pod 상태가 정상일 때만 `RUN_HEAVY_AUDIT = True`로 바꾸고 실행하세요.

결과를 이용해 고정 top-1000이 아니라 다음을 결정합니다.

- 값 binning이 안정적인 최소 numeric row 수
- 최소 환자 수
- 선택 threshold가 전체 numeric measurement의 몇 %를 보존하는지

In [ ]:
if not RUN_HEAVY_AUDIT:
    print("Skipped. Set RUN_HEAVY_AUDIT = True in the configuration cell first.")
else:
    lab_pair_frequency = query_df(sql.SQL("""
    WITH train_person AS (
        SELECT person_id
        FROM {}.person
        WHERE mod(
            hashtextextended(person_id::text, %s)
                & 9223372036854775807,
            100
        ) < 70
    )
    SELECT
        m.measurement_concept_id,
        m.unit_concept_id,
        COUNT(*) AS numeric_rows,
        COUNT(DISTINCT m.person_id) AS patients,
        MIN(m.measurement_date)::text AS first_date,
        MAX(m.measurement_date)::text AS last_date
    FROM {}.measurement AS m
    JOIN train_person AS p USING (person_id)
    WHERE m.value_as_number IS NOT NULL
      AND m.measurement_date BETWEEN DATE '1900-01-01' AND %s::date
      AND m.measurement_concept_id IS NOT NULL
      AND m.measurement_concept_id <> 0
    GROUP BY m.measurement_concept_id, m.unit_concept_id
    ORDER BY numeric_rows DESC
    """).format(
        sql.Identifier(DB_SCHEMA),
        sql.Identifier(DB_SCHEMA),
    ), (SPLIT_SEED, DB_END_DATE))

    lab_pair_frequency.to_parquet(
        OUTPUT_DIR / "train_numeric_lab_pair_frequency.parquet",
        index=False,
    )
    display(lab_pair_frequency.head(30))

    total_numeric = lab_pair_frequency.numeric_rows.sum()
    coverage_rows = []
    for min_rows in [30, 100, 300, 1000, 3000, 10000]:
        selected = lab_pair_frequency.numeric_rows >= min_rows
        coverage_rows.append({
            "min_numeric_rows": min_rows,
            "selected_pairs": int(selected.sum()),
            "selected_patients_max_not_additive": int(
                lab_pair_frequency.loc[selected, "patients"].max()
                if selected.any() else 0
            ),
            "numeric_row_coverage": float(
                lab_pair_frequency.loc[selected, "numeric_rows"].sum()
                / total_numeric
            ),
        })
    lab_threshold_coverage = pd.DataFrame(coverage_rows)
    display(lab_threshold_coverage)
    lab_threshold_coverage.to_csv(
        OUTPUT_DIR / "lab_threshold_coverage.csv",
        index=False,
    )


## 7. Categorical LAB 분포

Heavy audit를 켠 경우에만 train split의 categorical measurement 조합을 집계합니다.

In [ ]:
if not RUN_HEAVY_AUDIT:
    print("Skipped. Set RUN_HEAVY_AUDIT = True first.")
else:
    categorical_lab_frequency = query_df(sql.SQL("""
    WITH train_person AS (
        SELECT person_id
        FROM {}.person
        WHERE mod(
            hashtextextended(person_id::text, %s)
                & 9223372036854775807,
            100
        ) < 70
    )
    SELECT
        m.measurement_concept_id,
        m.value_as_concept_id,
        COUNT(*) AS rows,
        COUNT(DISTINCT m.person_id) AS patients
    FROM {}.measurement AS m
    JOIN train_person AS p USING (person_id)
    WHERE m.value_as_concept_id IS NOT NULL
      AND m.value_as_concept_id <> 0
      AND m.measurement_date BETWEEN DATE '1900-01-01' AND %s::date
    GROUP BY m.measurement_concept_id, m.value_as_concept_id
    ORDER BY rows DESC
    """).format(
        sql.Identifier(DB_SCHEMA),
        sql.Identifier(DB_SCHEMA),
    ), (SPLIT_SEED, DB_END_DATE))
    categorical_lab_frequency.to_parquet(
        OUTPUT_DIR / "train_categorical_lab_frequency.parquet",
        index=False,
    )
    display(categorical_lab_frequency.head(30))


## 8. 실행 manifest 저장

In [ ]:
manifest = {
    "db_host": DB_HOST,
    "db_port": DB_PORT,
    "db_name": DB_NAME,
    "db_schema": DB_SCHEMA,
    "db_end_date": DB_END_DATE,
    "split_seed": SPLIT_SEED,
    "split_rule": "hashtextextended(person_id::text, seed) mod 100; 70/15/15",
    "run_heavy_audit": RUN_HEAVY_AUDIT,
    "python_version": platform.python_version(),
    "logical_cpu_count": psutil.cpu_count(logical=True),
    "ram_gb": round(psutil.virtual_memory().total / 1024**3, 2),
}
with open(OUTPUT_DIR / "manifest.json", "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

print(json.dumps(manifest, indent=2))
print("Saved outputs:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name, f"({path.stat().st_size / 1024**2:.2f} MB)")


In [ ]:
conn.close()
DB_PASSWORD = None
print("Database connection closed and password variable cleared.")
